In [ ]:
# ───────────────────────── 0. CONFIGURACIÓN ─────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "selectgo_fase2"
RESET = True

# Host IPv4 del Pooler de Supabase
PG_HOST = "aws-0-us-west-2.pooler.supabase.com"
PG_PORT = 6543

# Usuario completo con sufijo del proyecto
PG_USER = "postgres.yqylevwobpxeyldcwkuc"
PG_PASSWORD = dbutils.secrets.get(scope="selectgo", key="supabase_pw")

# URL JDBC con prepareThreshold=0
JDBC_URL = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/postgres?sslmode=require&prepareThreshold=0"


def read_pg(table_or_query: str):
  # Si es un nombre de tabla directo (ej. "public.venta"), se pasa directamente;
  # si es una consulta SQL con espacios, se envuelve en subconsulta con alias.
  dbtable = (
      table_or_query
      if " " not in table_or_query.strip()
      else f"({table_or_query}) AS subq"
  )

  return (
      spark.read.format("jdbc")
      .option("url", JDBC_URL)
      .option("dbtable", dbtable)
      .option("user", PG_USER)
      .option("password", PG_PASSWORD)
      .option("driver", "org.postgresql.Driver")
      .option("prepareThreshold", "0")
      .load()
  )
  
T_VENTAS_RAW = f"{CATALOG}.bronze.ventas_raw"
T_MOV_RAW = f"{CATALOG}.bronze.movimientos_inventario_raw"
T_EXT_RAW = f"{CATALOG}.bronze.market_reference_raw"
T_VENTAS_CL = f"{CATALOG}.silver.ventas_clean"
T_VENTAS_QT = f"{CATALOG}.silver.ventas_quarantine"
T_MOV_CL = f"{CATALOG}.silver.movimientos_clean"
T_MOV_QT = f"{CATALOG}.silver.movimientos_quarantine"
T_BENCHMARK = f"{CATALOG}.silver.price_benchmark_external"
T_DIM_PROD = f"{CATALOG}.gold.dim_producto"
T_DIM_TIENDA = f"{CATALOG}.gold.dim_tienda"
T_DIM_CANAL = f"{CATALOG}.gold.dim_canal"
T_DIM_FECHA = f"{CATALOG}.gold.dim_fecha"
T_FACT_VENT = f"{CATALOG}.gold.fact_ventas"
T_INVENTARIO = f"{CATALOG}.gold.inventario_actual"